# Entraînement des modèles — Prédiction fin d'alerte foudre

Notebook d'entraînement complet pour Kaggle T4 x2.  
Lance successivement tous les modèles du repo `prediction-orage` :

| Phase | Modèle | Script |
|-------|--------|---------|
| 1 | Baseline 30 min + XGBoost survival:AFT | `evaluation.py` |
| 1 | BNN (MC Dropout sur MLP) | `bnn_model.py` |
| 2 | Hawkes Classique + Neural Hawkes GRU V1 | `hawkes_models.py` |
| 2bis | Neural Hawkes V2 (GRU v2, TF, TF-Large, par aéroport) | `neural_hawkes_v2.py` |
| 2bis | Neural Hawkes V3 Gaussien — **multi-GPU** | `train_parallel.py` |
| 2bis | Hawkes Bayésien (MC Dropout + Variationnel) | `bayesian_hawkes.py` |
| 2ter | Spatial MoE (3 experts, 20 features) | `spatial_moe_model.py` |

**Avant de lancer** : ajouter le dataset CSV comme *Input Dataset* dans Kaggle  
puis mettre à jour `DATA_CSV_INPUT` dans la cellule de configuration ci-dessous.

## 0. Setup

In [ ]:
!pip install -q xgboost tqdm scipy

In [ ]:
import subprocess
import sys
import shutil
from pathlib import Path

# ⚙️  CONFIGURATION — adapter selon le nom du dataset ajouté en Input
DATA_CSV_INPUT = "/kaggle/input/prediction-orage-data/segment_alerts_all_airports_train.csv"

REPO_DIR   = Path("/kaggle/working/prediction-orage")
OUTPUT_DIR = Path("/kaggle/working/outputs")

# Clonage ou mise à jour — git pull si le dossier existe déjà (session réutilisée)
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "-b", "reproduction",
         "https://github.com/Lucas64000/prediction-orage.git",
         str(REPO_DIR)],
        check=True,
    )
else:
    # Le dossier existe déjà : on tire les derniers commits pour avoir le code à jour
    subprocess.run(["git", "pull"], cwd=str(REPO_DIR), check=True)

# Dossiers attendus par les modules (chemins relatifs à __file__)
(REPO_DIR / "data").mkdir(exist_ok=True)
(REPO_DIR / "models").mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Copie du CSV depuis le dataset Kaggle vers data/
DATA_CSV_DEST = REPO_DIR / "data" / "segment_alerts_all_airports_train.csv"
if not DATA_CSV_DEST.exists():
    shutil.copy(DATA_CSV_INPUT, DATA_CSV_DEST)

SRC_DIR = str(REPO_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"Repo    : {REPO_DIR}")
print(f"CSV     : {DATA_CSV_DEST} (exists={DATA_CSV_DEST.exists()})")
print(f"Outputs : {OUTPUT_DIR}")

## 1. Chargement des données et features de survie

Les modèles séquentiels (Hawkes, V2, V3, Bayesian, MoE) travaillent directement  
sur les éclairs bruts. XGBoost et BNN ont besoin des features tabulaires pre-calculées  
(`features_survival.parquet`).

In [ ]:
import data_loader
import features as feat_module

print("Chargement du CSV...")
df     = data_loader.load_raw()          # lit depuis data/segment_alerts_all_airports_train.csv
alerts = data_loader.load_alerts(df)
print(f"  {len(df):,} éclairs chargés, {len(alerts):,} en session d'alerte")

FEATURES_PARQUET = REPO_DIR / "data" / "features_survival.parquet"
if not FEATURES_PARQUET.exists():
    print("Construction des features de survie (XGBoost / BNN)...")
    feat_df = feat_module.build_features(alerts)
    feat_df.to_parquet(FEATURES_PARQUET, index=False)
    print(f"  {len(feat_df):,} lignes sauvegardées → {FEATURES_PARQUET}")
else:
    print(f"  Features déjà présentes : {FEATURES_PARQUET}")

## 2. Phase 1 — Baseline 30 min & XGBoost survival:AFT

La **baseline** est la règle métier actuelle : prédire systématiquement 30 minutes restantes.  
**XGBoost AFT** (Accelerated Failure Time) modélise ln(T) sur les 20 features tabulaires.

In [ ]:
import evaluation

print("Chargement des features de survie...")
feat_df = evaluation.load_features()

print("Split temporel (GroupShuffleSplit par session)...")
train_tab, val_tab = evaluation.temporal_split(feat_df)

# Baseline
val_tab = evaluation.baseline_30min(val_tab)
baseline_metrics = evaluation.compute_metrics(val_tab, "pred_baseline", "Baseline 30 min")

# XGBoost
xgb_model, val_tab = evaluation.train_xgboost_survival(train_tab, val_tab)
xgb_metrics = evaluation.compute_metrics(val_tab, "pred_xgb_aft", "XGBoost survival:AFT")

xgb_model.save_model(str(REPO_DIR / "models" / "xgboost_aft.json"))
print("Modèle sauvegardé : models/xgboost_aft.json")

## 3. Phase 1 — BNN (Monte Carlo Dropout sur MLP)

MLP 3 couches avec dropout actif à l'inférence → N passes forward = distribution de prédictions.  
Fournit une **incertitude calibrée** (cible : ~68 % des vrais dans 1σ).

In [ ]:
import bnn_model

print("Chargement et split...")
train_bnn, val_bnn = bnn_model.load_and_split()
print(f"  Train : {len(train_bnn):,} | Val : {len(val_bnn):,}")

bnn, scaler, y_pred_bnn, y_std_bnn, y_true_bnn = bnn_model.train_bnn(
    train_bnn, val_bnn, n_epochs=80
)
print("Modèle sauvegardé : models/bnn_mc_dropout.pt")

## 4. Phase 2 — Hawkes Classique & Neural Hawkes GRU V1

**Hawkes Classique** : MLE sur le kernel exponentiel `λ(t) = μ + α·β·Σexp(−β·(t−tᵢ))`.  
**Neural Hawkes GRU V1** : GRU (hidden=32) apprend la dynamique d'intensité, 4 features par éclair.

In [ ]:
import hawkes_models

print("Préparation des sessions Hawkes (4 features)...")
sessions_hawkes = hawkes_models.prepare_hawkes_sessions(alerts)
print(f"  {len(sessions_hawkes)} sessions")

hawkes_classique, neural_hawkes_v1 = hawkes_models.evaluate_hawkes_models(sessions_hawkes)

# Sauvegarde
import json, torch
params = {"mu": hawkes_classique.mu, "alpha": hawkes_classique.alpha, "beta": hawkes_classique.beta}
with open(str(REPO_DIR / "models" / "hawkes_classique_params.json"), "w") as f:
    json.dump(params, f, indent=2)
torch.save(neural_hawkes_v1.model.state_dict(), str(REPO_DIR / "models" / "neural_hawkes_gru_v1.pt"))
print("Modèles sauvegardés : models/hawkes_classique_params.json, neural_hawkes_gru_v1.pt")

## 5. Phase 2bis — Neural Hawkes V2 (multi-tâche, 12 features)

4 variantes entraînées séquentiellement sur le GPU disponible :
- **V1** GRU v2 (2 couches, hidden=64)
- **V2** Transformer (d=64, 4 têtes, 3 couches)
- **V3** Transformer Large (d=128, 8 têtes, 4 couches)
- **V4** Transformers par aéroport (modèles spécialisés)

Tâche conjointe : NLL processus ponctuel + régression `time_to_end`.

In [ ]:
import neural_hawkes_v2

print("Préparation des sessions V2 (12 features)...")
sessions_v2 = neural_hawkes_v2.prepare_sessions_v2(alerts)
print(f"  {len(sessions_v2)} sessions, dim={sessions_v2[0]['features'].shape[1]}")

results_v2, trainer_tf_v2, trainer_tf_lg_v2 = neural_hawkes_v2.evaluate_all_variants(sessions_v2)

print("\nRésultats V2 :")
for key, r in results_v2.items():
    print(f"  {key:35s} | MAE={r['mae']:.2f} | Biais={r['bias']:+.2f}")

## 6. Phase 2bis — Neural Hawkes V3 Gaussien (multi-GPU)

Version avec sortie **gaussienne** en log-espace `(μ, log σ)` pour des incertitudes calibrées.  
Le dispatcher `train_parallel.run_v3()` répartit automatiquement les variantes sur les **2 T4** :
- **Vague 1** (parallèle) : V3a GRU, V3b TF, V3c TF-Large
- **Vague 2** (parallèle) : V3d un Transformer par aéroport

Résultats, poids et erreurs sauvegardés dans `outputs/`.

In [ ]:
import train_parallel

# run_v3 détecte les GPU disponibles, prépare les données en interne et
# dispatch les variantes sur les devices en parallèle.
results_v3 = train_parallel.run_v3(
    data_csv=str(DATA_CSV_DEST),
    output_dir=str(OUTPUT_DIR),
    n_epochs=80,
    seed=42,
)

## 7. Phase 2bis — Hawkes Bayésien

Deux approches d'inférence bayésienne sur le Transformer Hawkes (12 features, multi-tâche) :
- **Option A — MC Dropout** : dropout actif à l'inférence, N passes → `(mean, std)`
- **Option B — Variationnel** : couches linéaires variationnelles, optimisation ELBO

In [ ]:
import bayesian_hawkes

# sessions_v2 déjà préparées à la cellule 5 — les deux partagent le même format 12-D
results_bayes, trainer_mc, trainer_var = bayesian_hawkes.run_bayesian_experiments(sessions_v2)

print("\nRésultats Bayésiens :")
for key, r in results_bayes.items():
    cal = f"{r.get('calibration_1std', 0)*100:.0f}%" if "calibration_1std" in r else "---"
    print(f"  {key:40s} | MAE={r['mae']:.2f} | Biais={r['bias']:+.2f} | 1σ={cal}")

## 8. Phase 2ter — Spatial MoE (Mixture of Experts)

Transformer partagé + 3 experts spécialisés selon la dynamique spatiale de l'orage :
- **Expert A** Orage qui s'approche
- **Expert B** Orage qui s'éloigne → meilleure précision de fin d'alerte
- **Expert C** Situation intermédiaire

Entrée : **20 features** (12 de V2 + 8 spatiales : vélocité radiale, étalement angulaire, etc.)  
Sortie : mélange gaussien `(μ_mix, σ_mix)` en log-espace.

In [ ]:
import spatial_moe_model
from spatial_features import prepare_sessions_spatial

print("Préparation des sessions spatiales (20 features)...")
sessions_spatial = prepare_sessions_spatial(alerts)
print(f"  {len(sessions_spatial)} sessions, dim={sessions_spatial[0]['features'].shape[1]}")

result_moe, trainer_moe = spatial_moe_model.run_moe_experiment(sessions_spatial)

if result_moe:
    print(f"\nSpatial MoE → MAE={result_moe['mae']:.2f} | Biais={result_moe['bias']:+.2f} | P90={result_moe.get('p90', 0):.2f}")

## 9. Résumé comparatif

## 9. Évaluation risque / gain (θ = 0.4)

Pour chaque modèle, on calcule à θ = 0.4 :
- **Gain** = Σ (last_lightning + 30 min − predicted_end) sur les alertes couvertes
- **Risque** = éclairs < 3 km après predicted_end / total éclairs < 3 km (cible < 2 %)

On ne garde que les prédictions avec confidence ≥ 0.4, i.e. le modèle prédit < 18 min restantes.  
Pour les modèles gaussiens (V3, Bayesian) la confiance peut aussi être calculée via P(pred < 30 | μ, σ).

In [ ]:
import risk_evaluation as re_eval

THETA = 0.4

risk_results = {}

# ── Baseline 30 min ────────────────────────────────────────────────────────────
# La baseline prédit toujours last_known_lightning + 30 min → gain = 0 par définition
# On l'inclut pour avoir la référence sur le risque.
preds_baseline = re_eval.predictions_from_tabular(val_tab, "pred_baseline")
risk_results["Baseline 30 min"] = re_eval.evaluate_at_theta(preds_baseline, alerts, theta=THETA)

# ── XGBoost AFT ───────────────────────────────────────────────────────────────
preds_xgb = re_eval.predictions_from_tabular(val_tab, "pred_xgb_aft")
risk_results["XGBoost AFT"] = re_eval.evaluate_at_theta(preds_xgb, alerts, theta=THETA)

# ── BNN MC Dropout ────────────────────────────────────────────────────────────
val_bnn_with_pred = val_bnn.copy()
val_bnn_with_pred["pred_bnn"] = y_pred_bnn
preds_bnn = re_eval.predictions_from_tabular(val_bnn_with_pred, "pred_bnn")
risk_results["BNN MC Dropout"] = re_eval.evaluate_at_theta(preds_bnn, alerts, theta=THETA)

# ── Neural Hawkes V2 ──────────────────────────────────────────────────────────
for key, r in results_v2.items():
    if r is None or r.get("errors_df") is None:
        continue
    preds = re_eval.predictions_from_errors_df(r["errors_df"], alerts)
    risk_results[f"V2 {key}"] = re_eval.evaluate_at_theta(preds, alerts, theta=THETA)

# ── Neural Hawkes V3 (multi-GPU) ───────────────────────────────────────────────
for key in ["V3a GRU", "V3b TF", "V3c TF-Large", "V3d Ensemble"]:
    r = results_v3.get(key)
    if r is None or r.get("errors_df") is None:
        continue
    use_unc = "uncertainty" in r["errors_df"].columns
    preds = re_eval.predictions_from_errors_df(r["errors_df"], alerts, use_uncertainty=use_unc)
    risk_results[key] = re_eval.evaluate_at_theta(preds, alerts, theta=THETA)

# ── Hawkes Bayésien ───────────────────────────────────────────────────────────
for key, r in results_bayes.items():
    if r is None or r.get("errors_df") is None:
        continue
    use_unc = "uncertainty" in r["errors_df"].columns
    preds = re_eval.predictions_from_errors_df(r["errors_df"], alerts, use_uncertainty=use_unc)
    risk_results[f"Bayes {key}"] = re_eval.evaluate_at_theta(preds, alerts, theta=THETA)

# ── Spatial MoE ───────────────────────────────────────────────────────────────
if result_moe and result_moe.get("errors_df") is not None:
    use_unc = "uncertainty" in result_moe["errors_df"].columns
    preds = re_eval.predictions_from_errors_df(result_moe["errors_df"], alerts, use_uncertainty=use_unc)
    risk_results["Spatial MoE"] = re_eval.evaluate_at_theta(preds, alerts, theta=THETA)

re_eval.print_risk_table(risk_results, theta=THETA)

In [ ]:
import numpy as np

rows = []

# Phase 1
rows.append({"Phase": "1", "Modèle": "Baseline 30 min",
             "MAE": baseline_metrics["mae"], "RMSE": baseline_metrics["rmse"],
             "Biais": baseline_metrics["bias"]})
rows.append({"Phase": "1", "Modèle": "XGBoost survival:AFT",
             "MAE": xgb_metrics["mae"], "RMSE": xgb_metrics["rmse"],
             "Biais": xgb_metrics["bias"]})

# BNN
bnn_mae  = float(np.mean(np.abs(y_true_bnn - y_pred_bnn)))
bnn_rmse = float(np.sqrt(np.mean((y_true_bnn - y_pred_bnn)**2)))
bnn_bias = float(np.mean(y_pred_bnn - y_true_bnn))
rows.append({"Phase": "1", "Modèle": "BNN MC Dropout (MLP)",
             "MAE": bnn_mae, "RMSE": bnn_rmse, "Biais": bnn_bias})

# V2
for key, r in results_v2.items():
    rows.append({"Phase": "2bis", "Modèle": f"V2 {key}",
                 "MAE": r["mae"], "RMSE": r.get("rmse", float("nan")),
                 "Biais": r["bias"]})

# V3
for key in ["V3a GRU", "V3b TF", "V3c TF-Large", "V3d Ensemble"]:
    if key in results_v3:
        r = results_v3[key]
        rows.append({"Phase": "2bis V3", "Modèle": key,
                     "MAE": r["mae"], "RMSE": r.get("rmse", float("nan")),
                     "Biais": r["bias"]})

# Bayésien
for key, r in results_bayes.items():
    rows.append({"Phase": "2bis", "Modèle": f"Bayes {key}",
                 "MAE": r["mae"], "RMSE": r.get("rmse", float("nan")),
                 "Biais": r["bias"]})

# MoE
if result_moe:
    rows.append({"Phase": "2ter", "Modèle": "Spatial MoE",
                 "MAE": result_moe["mae"], "RMSE": result_moe.get("rmse", float("nan")),
                 "Biais": result_moe["bias"]})

import pandas as pd
summary_df = pd.DataFrame(rows).set_index(["Phase", "Modèle"])
summary_df = summary_df.round(2).sort_values("MAE")

print("\n" + "="*65)
print("  RÉSUMÉ COMPARATIF — toutes phases")
print("="*65)
print(summary_df.to_string())

# Export
summary_df.to_csv(OUTPUT_DIR / "results_summary.csv")
print(f"\nTableau exporté : {OUTPUT_DIR / 'results_summary.csv'}")